In [1]:
import requests
import zipfile
import io
import pandas as pd

def download_unzip_and_read_excel(url, output_folder='.'):
    """
    Download a zip file from URL, unzip it, and read the Excel file.
    
    Args:
        url (str): URL of the zip file to download
        output_folder (str): Folder to save the extracted files (default: current directory)
    
    Returns:
        pandas.DataFrame: Contents of the Excel file
    """
    try:
        print(f"Downloading zip file from {url}...")
        # Download the zip file
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Unzip the file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Find the Excel file in the zip (assuming there's only one)
            excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls', '.csv'))]
            
            if not excel_files:
                raise ValueError("No Excel/CSV file found in the zip archive")
            
            # We'll use the first Excel file found
            excel_filename = excel_files[0]
            print(f"Found file in zip: {excel_filename}")
            
            # Extract the file (either to memory or disk)
            if output_folder:
                print(f"Extracting to {output_folder}...")
                zip_ref.extractall(output_folder)
                file_path = f"{output_folder}/{excel_filename}"
                print(f"Reading Excel file from {file_path}...")
                
                # Read the file based on extension
                if excel_filename.endswith('.csv'):
                    df = pd.read_csv(file_path)
                else:  # .xlsx or .xls
                    df = pd.read_excel(file_path)
            else:
                # Extract to memory
                with zip_ref.open(excel_filename) as file:
                    print("Reading Excel file from memory...")
                    if excel_filename.endswith('.csv'):
                        df = pd.read_csv(file)
                    else:  # .xlsx or .xls
                        df = pd.read_excel(file)
            
            print("Successfully read the Excel file!")
            return df
    
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip file")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

def download_unzip_and_read_excel_income(url, output_folder='.',name_sheet='ENSEMBLE'):
    """
    Download a zip file from URL, unzip it, and read the Excel file.
    
    Args:
        url (str): URL of the zip file to download
        output_folder (str): Folder to save the extracted files (default: current directory)
    
    Returns:
        pandas.DataFrame: Contents of the Excel file
    """
    try:
        print(f"Downloading zip file from {url}...")
        # Download the zip file
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Unzip the file in memory
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Find the Excel file in the zip (assuming there's only one)
            excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls', '.csv'))]
            
            if not excel_files:
                raise ValueError("No Excel/CSV file found in the zip archive")
            
            # We'll use the first Excel file found
            excel_filename = [f for f in excel_files if "DISP_COM" in f][0]
            print(f"Found file in zip: {excel_filename}")
            
            # Extract the file (either to memory or disk)
            if output_folder:
                print(f"Extracting to {output_folder}...")
                zip_ref.extractall(output_folder)
                file_path = f"{output_folder}/{excel_filename}"
                print(f"Reading Excel file from {file_path}...")
                
                # Read the file based on extension
                if excel_filename.endswith('.csv'):
                    df = pd.read_csv(file_path,error_bad_lines=False,sep=";")
                else:  # .xlsx or .xls
                    try:
                        df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5)
                    except:
                        df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5,engine='openpyxl')
            else:
                # Extract to memory
                with zip_ref.open(excel_filename) as file:
                    print("Reading Excel file from memory...")
                    if excel_filename.endswith('.csv'):
                        df = pd.read_csv(file,error_bad_lines=False,sep=";")
                    else:  # .xlsx or .xls
                        try:
                            df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5)
                        except:
                            df = pd.read_excel(file_path,sheet_name=name_sheet,skiprows=5,engine='openpyxl')
            
            print("Successfully read the Excel file!")
            return df
    
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
    except zipfile.BadZipFile:
        print("Error: The downloaded file is not a valid zip file")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

## Données vote législatives

In [3]:
# URL of the zip file
full_df = pd.DataFrame()
years = [2022, 2017, 2012, 2007, 2002]
for year in years:

    zip_url = f"https://conflit-politique-data.ams3.cdn.digitaloceanspaces.com/zip/leg{year}_csv.zip"

    # Download, unzip and read the Excel file
    data_frame = download_unzip_and_read_excel(zip_url, output_folder='../data/')
    common_columns = ['dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits','votants', 'exprimes','pvoteG', 'pvoteCG','pvoteC', 'pvoteCD', 'pvoteD', 'pvoteTG', 'pvoteTD', 'pvoteGCG','pvoteDCD']
    data_frame = data_frame[common_columns]
    data_frame['Year']  = year
    full_df = pd.concat([full_df, data_frame], ignore_index=True)
    print(f"Data for year {year} added to the DataFrame.")

full_df = full_df.set_index(['Year', 'dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits', 'votants', 'exprimes']).stack().reset_index()
full_df.columns = ['Year', 'dep', 'nomdep', 'codecommune', 'nomcommune', 'inscrits', 'votants', 'exprimes', 'vote_type', 'share']
full_df['vote_type'] = full_df['vote_type'].apply(lambda x: x.replace('pvote', ''))
full_df['absention'] = 1 - full_df['votants'] / full_df['inscrits']



Found file in zip: leg2022_csv/leg2022comm.csv
Extracting to ../data/...
Reading Excel file from ../data//leg2022_csv/leg2022comm.csv...


C:\Users\colin\AppData\Local\Temp\ipykernel_5960\2342353075.py:44: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Successfully read the Excel file!
Data for year 2022 added to the DataFrame.
Found file in zip: leg2017_csv/leg2017comm.csv
Extracting to ../data/...
Reading Excel file from ../data//leg2017_csv/leg2017comm.csv...


C:\Users\colin\AppData\Local\Temp\ipykernel_5960\2342353075.py:44: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Successfully read the Excel file!
Data for year 2017 added to the DataFrame.
Found file in zip: leg2012_csv/leg2012comm.csv
Extracting to ../data/...
Reading Excel file from ../data//leg2012_csv/leg2012comm.csv...


...

Successfully read the Excel file!
Data for year 2012 added to the DataFrame.
Found file in zip: leg2007_csv/leg2007comm.csv
Extracting to ../data/...
Reading Excel file from ../data//leg2007_csv/leg2007comm.csv...


C:\Users\colin\AppData\Local\Temp\ipykernel_5960\2342353075.py:44: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Successfully read the Excel file!
Data for year 2007 added to the DataFrame.
Found file in zip: leg2002_csv/leg2002comm.csv
Extracting to ../data/...
Reading Excel file from ../data//leg2002_csv/leg2002comm.csv...


C:\Users\colin\AppData\Local\Temp\ipykernel_5960\2342353075.py:44: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Successfully read the Excel file!
Data for year 2002 added to the DataFrame.


In [11]:
full_df

,Year,dep,nomdep,codecommune,nomcommune,inscrits,votants,exprimes,vote_type,share,absention
0,2022,1,AIN,1001,L'ABERGEMENT-CLÉMENCIAT,644,343,339.0,G,0.165192,0.467391
1,2022,1,AIN,1001,L'ABERGEMENT-CLÉMENCIAT,644,343,339.0,CG,0.070796,0.467391
2,2022,1,AIN,1001,L'ABERGEMENT-CLÉMENCIAT,644,343,339.0,C,0.244838,0.467391
3,2022,1,AIN,1001,L'ABERGEMENT-CLÉMENCIAT,644,343,339.0,CD,0.200590,0.467391
4,2022,1,AIN,1001,L'ABERGEMENT-CLÉMENCIAT,644,343,339.0,D,0.318584,0.467391
...,...,...,...,...,...,...,...,...,...,...,...
1620085,2002,95,VAL-D'OISE,95690,WY-DIT-JOLI-VILLAGE,252,202,200.0,D,0.140000,0.198413
1620086,2002,95,VAL-D'OISE,95690,WY-DIT-JOLI-VILLAGE,252,202,200.0,TG,0.275000,0.198413
1620087,2002,95,VAL-D'OISE,95690,WY-DIT-JOLI-VILLAGE,252,202,200.0,TD,0.725000,0.198413
1620088,2002,95,VAL-D'OISE,95690,WY-DIT-JOLI-VILLAGE,252,202,200.0,GCG,0.275000,0.198413


[Data 2021](https://www.insee.fr/fr/statistiques/7756855?sommaire=7756859&q=Dispositif+Fichier+localis%C3%A9%20social+et+fiscal+(Filosofi)), [Data 2017](https://www.insee.fr/fr/statistiques/4291712#consulter),[Data 2012](https://www.insee.fr/fr/statistiques/2043745)

In [6]:
full_df.to_csv('../data/legislatives.csv', index=False)

In [11]:
urls = ["https://www.insee.fr/fr/statistiques/fichier/2043745/indic-struct-distrib-revenu-communes-2012.zip",
        "https://www.insee.fr/fr/statistiques/fichier/4291712/indic-struct-distrib-revenu-2017-COMMUNES.zip",
        "https://www.insee.fr/fr/statistiques/fichier/7756855/indic-struct-distrib-revenu-2021-COMMUNES_csv.zip"]

urls = ["https://www.insee.fr/fr/statistiques/fichier/2043745/indic-struct-distrib-revenu-communes-2012.zip",
        "https://www.insee.fr/fr/statistiques/fichier/4291712/indic-struct-distrib-revenu-2017-COMMUNES.zip"]

years = [2012,2017, 2021]
full_rev = pd.DataFrame()
for i in range(len(urls)):
    zip_url = urls[i]
    rev = download_unzip_and_read_excel_income(zip_url, name_sheet='ENSEMBLE', output_folder='../data/')
    year = years[i]
    rev = rev[['CODGEO', f'NBMEN{str(year)[2:]}', f'NBPERS{str(year)[2:]}', f'NBUC{str(year)[2:]}',
                   f'Q2{str(year)[2:]}', f'GI{str(year)[2:]}']]
    rev.columns = ['CODGEO','Menages','Population','UC','Med_Niveau_vie','Gini']
    rev['Year'] = str(year)
    full_rev = pd.concat([full_rev, rev], ignore_index=True)


Found file in zip: indic-struct-distrib-revenu-communes-2012/FILO_DISP_COM.xls
Extracting to ../data/...
Reading Excel file from ../data//indic-struct-distrib-revenu-communes-2012/FILO_DISP_COM.xls...
Successfully read the Excel file!
Found file in zip: FILO2017_DISP_COM.xlsx
Extracting to ../data/...
Reading Excel file from ../data//FILO2017_DISP_COM.xlsx...
Successfully read the Excel file!


In [15]:
df_diplome = pd.read_excel('C:/Users/colin/Downloads/base-cc-diplomes-formation-2020_xlsx/base-cc-diplomes-formation-2020.xlsx', sheet_name='COM_2020', skiprows=5)


In [16]:
var_list = ['CODGEO', 'P20_NSCOL15P', 'P20_NSCOL15P_BAC','P20_NSCOL15P_SUP5']
df_diplome = df_diplome[var_list]
df_diplome = df_diplome.rename(columns={'P20_NSCOL15P': 'Pop_non_scola', 'P20_NSCOL15P_BAC': 'Pop_bac', 'P20_NSCOL15P_SUP5': 'Pop_Sup5'})
df_diplome['Pop_bac'] = df_diplome['Pop_bac']/df_diplome['Pop_non_scola']
df_diplome['Pop_Sup5'] = df_diplome['Pop_Sup5']/df_diplome['Pop_non_scola']

In [19]:
full_rev=full_rev[['CODGEO', 'Menages', 'Population', 'UC', 'Med_Niveau_vie', 'Gini', 'Year']]
full_rev = full_rev.merge(df_diplome, on='CODGEO', how='left')

In [20]:
full_rev

,CODGEO,Menages,Population,UC,Med_Niveau_vie,Gini,Year,Pop_non_scola,Pop_bac,Pop_Sup5
0,01001,299,780.5,511.75,22253.000000,NaN,2012,607.861825,0.201756,0.063676
1,01002,97,227.0,153.20,21765.714286,NaN,2012,181.901817,0.221732,0.131580
2,01004,5897,13420.0,9162.90,19236.666667,0.264337,2012,10493.934917,0.184745,0.069766
3,01005,615,1673.5,1080.15,21743.888889,NaN,2012,1322.000000,0.192890,0.065809
4,01006,48,111.0,76.70,20354.400000,NaN,2012,100.114035,0.158416,0.059406
...,...,...,...,...,...,...,...,...,...,...
64690,97420,8082,23798.0,14874.90,15210.000000,0.321000,2017,16431.158055,0.179487,0.045533
64691,97421,2433,7166.0,4481.40,11580.000000,0.310000,2017,5114.794469,0.129922,0.016684
64692,97422,29215,76686.0,49671.60,14460.000000,0.335000,2017,55379.721646,0.171035,0.039121
64693,97423,2411,7097.0,4473.60,14460.000000,0.307000,2017,5113.000000,0.160180,0.030902


In [21]:
full_rev.to_csv('../data/revenu.csv', index=False)

### Reelection municipale

In [ ]:
path = "C:/Users/colin/Downloads/candidats_results(1).parquet"
df_muni = pd.read_parquet(path)
df_muni = df_muni.sort_values('id_election')


In [4]:
df_muni_2020 = df_muni[df_muni.id_election.isin(['2020_muni_t2', '2020_muni_t1'])]
df_muni_2020['GEO_INSEE'] = df_muni_2020['Code du département'] + df_muni['Code de la commune']
df_muni_2020.sort_values('GEO_INSEE', inplace=True)
df_muni_2020 = df_muni_2020[['id_election', 'Nom', 'Nuance', 'Prénom', 'GEO_INSEE', 'Voix', 'Sexe']].groupby(['id_election', 'GEO_INSEE', 'Nuance', 'Nom', 'Prénom', 'Sexe']).sum().reset_index()
df_muni_2020['share'] = 100*df_muni_2020['Voix'] / df_muni_2020.groupby(['id_election', 'GEO_INSEE'])['Voix'].transform('sum')
# df_muni_lit['share'] = df_muni_lit['share'].round(2)


C:\Users\colin\AppData\Local\Temp\ipykernel_65844\2968335391.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2020['GEO_INSEE'] = df_muni_2020['Code du département'] + df_muni['Code de la commune']
C:\Users\colin\AppData\Local\Temp\ipykernel_65844\2968335391.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2020.sort_values('GEO_INSEE', inplace=True)


In [12]:
df_elected_t2 = df_muni_2020[df_muni_2020['id_election'] == '2020_muni_t2']
df_elected_t2 = df_elected_t2.dropna(subset=['share'])
df_elected_t2 = df_elected_t2.loc[df_elected_t2.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_t2
df_elected_t1 = df_muni_2020[~df_muni_2020.GEO_INSEE.isin(df_elected_t2.GEO_INSEE)]
df_elected_t1 = df_elected_t1.dropna(subset=['share'])
df_elected_t1 = df_elected_t1[df_elected_t1['id_election'] == '2020_muni_t1']
df_elected_t1 = df_elected_t1.loc[df_elected_t1.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_2020 = pd.concat([df_elected_t1, df_elected_t2], ignore_index=True)

df_elected_2020

,id_election,GEO_INSEE,Nuance,Nom,Prénom,Sexe,Voix,share
0,2020_muni_t1,01001,NC,BOUILLOUX,Delphine,F,267.0,6.964006
1,2020_muni_t1,01002,NC,ORSET,Max,M,123.0,9.468822
2,2020_muni_t1,01004,LDVC,FABRE,Daniel,M,1410.0,50.976139
3,2020_muni_t1,01005,LNC,PERNET,Pierre,M,315.0,100.000000
4,2020_muni_t1,01006,NC,GUILLOUT,Raymond,M,57.0,10.877863
...,...,...,...,...,...,...,...,...
34999,2020_muni_t2,ZP739,NC,VARUATUA,Euloge,M,73.0,46.202532
35000,2020_muni_t2,ZP742,NC,TAIEMOEARO,Tepepe,F,152.0,5.442177
35001,2020_muni_t2,ZP743,NC,IOANE,Théodore Taputuura,M,130.0,3.993856
35002,2020_muni_t2,ZP755,NC,BRANDER,Tevahineheipua,F,118.0,5.312922


In [ ]:
df_muni_2014 = df_muni[df_muni.id_election.isin(['2014_muni_t2', '2014_muni_t1'])]
df_muni_2014['GEO_INSEE'] = df_muni_2014['Code du département'] + df_muni['Code de la commune']
df_muni_2014.sort_values('GEO_INSEE', inplace=True)
df_muni_2014 = df_muni_2014.dropna(subset=['Voix'])
df_muni_2014 = df_muni_2014[['id_election', 'Nom', 'Nuance', 'Prénom', 'GEO_INSEE', 'Voix']].groupby(['id_election', 'GEO_INSEE', 'Nuance',
    'Nom', 'Prénom']).sum().reset_index()
df_muni_2014['share'] = 100*df_muni_2014['Voix'] / df_muni_2014.groupby(['id_election', 'GEO_INSEE'])['Voix'].transform('sum')


C:\Users\colin\AppData\Local\Temp\ipykernel_65844\1146686229.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2014['GEO_INSEE'] = df_muni_2014['Code du département'] + df_muni['Code de la commune']
C:\Users\colin\AppData\Local\Temp\ipykernel_65844\1146686229.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2014.sort_values('GEO_INSEE', inplace=True)


In [56]:
df_elected_t1 = df_muni_2014[df_muni_2014['id_election'] == '2014_muni_t1']
df_elected_t1 = df_elected_t1.dropna(subset=['share'])
df_elected_t1 = df_elected_t1.loc[df_elected_t1.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_t2 = df_muni_2014[~df_muni_2014.GEO_INSEE.isin(df_elected_t1.GEO_INSEE)]
df_elected_t2 = df_elected_t2.dropna(subset=['share'])
df_elected_t2 = df_elected_t2[df_elected_t2['id_election'] == '2014_muni_t2']
df_elected_t2 = df_elected_t2.loc[df_elected_t2.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_2014 = pd.concat([df_elected_t1, df_elected_t2], ignore_index=True)

In [15]:
df_muni_2008 = df_muni[df_muni.id_election.isin(['2008_muni_t2', '2008_muni_t1'])]
df_muni_2008['GEO_INSEE'] = df_muni_2008['Code du département'] + df_muni['Code de la commune']
df_muni_2008.sort_values('GEO_INSEE', inplace=True)
df_muni_2008 = df_muni_2008[['id_election', 'Nom', 'Nuance', 'Prénom', 'GEO_INSEE', 'Voix', 'Sexe']].groupby(['id_election', 'GEO_INSEE', 'Nuance',
    'Nom', 'Prénom', 'Sexe']).sum().reset_index()
df_muni_2008['share'] = 100*df_muni_2008['Voix'] / df_muni_2008.groupby(['id_election', 'GEO_INSEE'])['Voix'].transform('sum')

C:\Users\colin\AppData\Local\Temp\ipykernel_65844\862834064.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2008['GEO_INSEE'] = df_muni_2008['Code du département'] + df_muni['Code de la commune']
C:\Users\colin\AppData\Local\Temp\ipykernel_65844\862834064.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_muni_2008.sort_values('GEO_INSEE', inplace=True)


In [16]:
df_elected_t1 = df_muni_2008[df_muni_2008['id_election'] == '2008_muni_t1']
df_elected_t1 = df_elected_t1.dropna(subset=['share'])
df_elected_t1 = df_elected_t1.loc[df_elected_t1.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_t2 = df_muni_2008[~df_muni_2008.GEO_INSEE.isin(df_elected_t1.GEO_INSEE)]
df_elected_t2 = df_elected_t2.dropna(subset=['share'])
df_elected_t2 = df_elected_t2[df_muni_2008['id_election'] == '2008_muni_t2']
df_elected_t2 = df_elected_t2.loc[df_elected_t2.groupby(['GEO_INSEE'])['share'].idxmax()]
df_elected_2008 = pd.concat([df_elected_t1, df_elected_t2], ignore_index=True)

C:\Users\colin\AppData\Local\Temp\ipykernel_65844\944165782.py:6: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_elected_t2 = df_elected_t2[df_muni_2008['id_election'] == '2008_muni_t2']


In [57]:
df_elected_2008['Year'] = 2008
df_elected_2014['Year'] = 2014
df_elected_2020['Year'] = 2020
df_elected = pd.concat([df_elected_2008, df_elected_2014, df_elected_2020], ignore_index=True)

In [62]:
df_reelection = df_elected[df_elected['Year']==2020]
df_temp = df_elected[df_elected['Year']==2014]
df_temp['reelected_2020'] = 1
df_reelection = df_reelection.merge(df_temp[['GEO_INSEE', 'Nom', 'Prénom', 'reelected_2020']], on=['GEO_INSEE', 'Nom', 'Prénom'], how='left')
df_reelection['reelected_2020'] = df_reelection['reelected_2020'].fillna(0)

df_reelection_2014 = df_elected[df_elected['Year']==2014]
df_temp = df_elected[df_elected['Year']==2008]
df_temp['reelected_2014'] = 1
df_reelection_2014 = df_reelection_2014.merge(df_temp[['GEO_INSEE', 'Nom', 'Prénom', 'reelected_2014']], on=['GEO_INSEE', 'Nom', 'Prénom'], how='left')
df_reelection_2014['reelected_2014'] = df_reelection_2014['reelected_2014'].fillna(0)

df_reelection = df_reelection.merge(df_reelection_2014[['GEO_INSEE', 'reelected_2014']], on=['GEO_INSEE'], how='left')


C:\Users\colin\AppData\Local\Temp\ipykernel_65844\4044730725.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['reelected_2020'] = 1
C:\Users\colin\AppData\Local\Temp\ipykernel_65844\4044730725.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_temp['reelected_2014'] = 1


In [64]:
print(df_reelection.reelected_2020.sum()/df_reelection.shape[0] * 100, " % of reelected in 2020")
print(df_reelection.reelected_2014.sum()/df_reelection.shape[0] * 100, " % of reelected in 2014")
#print the number of nan meaning no matching in the merge
print(df_reelection[df_reelection.reelected_2014.isna()].shape[0], " % of reelected in 2014 not in the 2008 election")
print(df_reelection[df_reelection.reelected_2020.isna()].shape[0], " % of reelected in 2020 not in the 2014 election")

19.692035195977603  % of reelected in 2020
3.8367043766426696  % of reelected in 2014
49  % of reelected in 2014 not in the 2008 election
0  % of reelected in 2020 not in the 2014 election


In [104]:
df_reelection.to_csv('../data/reelection_muni.csv', index=False)